# Tool Retrieval under Distractor APIs with Memory-Based Recovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hemal0651/tool-retrieval-memory-recovery/blob/main/notebooks/tool_retrieval_memory_recovery.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**Authors:** Hemal Farouqe, Sheikh Mainul Islam, Tanjida Sultana Maria

---

### Abstract
When LLM agents have to choose a tool from catalogues replete with near-identical descriptions, i.e., distractor APIs, stateless selectors are prone to making the same error on like queries. To address this, we introduce a lightweight retrieve–select–execute loop backed by an **Experience Memory ($M$)**. This memory retains failed query–tool pairs and down-ranks those tools in keeping with query similarity:

$$\pi(q, t) = \max_{(q_i, t) \in M} \cos(f(q), f(q_i)) \quad (\text{gated by } \tau)$$
$$s'(q, t) = s(q, t) - \lambda \pi(q, t)$$

This notebook provides the complete reproducible pipeline evaluated on the ToolAlpaca benchmark (380 APIs, 2,695 tools, 2,643 single-tool queries) with +10/+20/+30 mined distractors, multi-split paired statistical testing, LLM selector experiments, and memory pollution analysis.


In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers datasets statsmodels groq matplotlib tqdm

# 2. Check GPU availability
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))


## 1. Minimal Working Example: The Look-Alike Distractor Problem
Let us first demonstrate the core failure mode of stateless retrieval using a minimal scenario where two tools share very similar descriptions, and show how Experience Memory recovers from repeated errors.


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

base_model = SentenceTransformer('all-MiniLM-L6-v2')

# Look-alike tools in banking domain
tools_demo = [
    'send_money_api: Transfers funds to a recipient account.',                # Ground truth
    'transfer_money_friend_api: Checks status of sending $50 to a friend.',   # Distractor
    'request_money_api: Sends a payment request to another user.'             # Other candidate
]

query_1 = 'Send $50 to my friend John'
query_2 = 'Send $20 to my friend Sarah'

# Compute standard cosine scores for Query 1
q1_emb = base_model.encode(query_1, convert_to_tensor=True, normalize_embeddings=True)
t_embs = base_model.encode(tools_demo, convert_to_tensor=True, normalize_embeddings=True)
scores_1 = q1_emb @ t_embs.T

print('=== Query 1 (Initial attempt) ===')
for i, tool in enumerate(tools_demo):
    print(f'Tool {i}: {scores_1[i].item():.4f} - {tool}')
chosen_1 = int(scores_1.argmax().item())
print(f'Stateless Selector chooses Tool {chosen_1} (Ground truth is Tool 0 -> FAILURE)')

# --- Store failure in Experience Memory M ---
memory_failures = {1: [q1_emb]} # Tool 1 failed on query_1

# Compute scores for Query 2 with Experience Memory (lambda=0.3, tau=0.6)
q2_emb = base_model.encode(query_2, convert_to_tensor=True, normalize_embeddings=True)
scores_2 = q2_emb @ t_embs.T
penalized_scores_2 = scores_2.clone()

# Calculate similarity-weighted penalty
sim_to_failed = (torch.stack(memory_failures[1]) @ q2_emb).max().item()
lam, tau = 0.3, 0.6
if sim_to_failed >= tau:
    penalized_scores_2[1] -= lam * sim_to_failed

print('\n=== Query 2 (With Experience Memory) ===')
for i, tool in enumerate(tools_demo):
    print(f'Tool {i}: Base={scores_2[i].item():.4f} -> Penalized={penalized_scores_2[i].item():.4f} - {tool}')
chosen_2 = int(penalized_scores_2.argmax().item())
print(f'Memory-Assisted Selector chooses Tool {chosen_2} (Ground truth is Tool 0 -> SUCCESS!)')


## 2. Dataset Preparation: ToolAlpaca Benchmark
We construct our controlled benchmark from the ToolAlpaca dataset (`train_data.json`).
- Each function across 380 APIs is indexed as an individual candidate tool (`Function_Name: Short_Description`).
- We filter for queries having exactly one ground truth tool in `intermediate_steps` (skipping multi-tool or empty tasks).
- Dataset splits are performed strictly at the **API level** (70% train, 15% validation, 15% test) to prevent cross-API leakage.


In [ ]:
import json, random, os, subprocess

# Clone ToolAlpaca if not already present
if not os.path.exists('ToolAlpaca'):
    !git clone -q --depth 1 https://github.com/tangqiaoyu/ToolAlpaca.git

data_path = 'ToolAlpaca/data/train_data.json'
with open(data_path, 'r', encoding='utf-8') as f:
    raw_alpaca = json.load(f)

# 1. Extract tools pool
tools = {}
for api in raw_alpaca:
    api_name = api.get('Name', '')
    for fname, desc in api.get('Function_Description', {}).items():
        short = desc.split('\nParameters:')[0].strip()
        tools[f'{api_name}::{fname}'] = f'{fname}: {short}'

# 2. Extract single-tool queries
data = []
skipped = 0
for api in raw_alpaca:
    api_name = api.get('Name', '')
    for inst in api.get('Instances', []):
        try:
            steps = inst.get('intermediate_steps', [])
            used = {s[0][0] for s in steps if s and s[0]}
        except Exception:
            used = set()
        if len(used) != 1:
            skipped += 1
            continue
        tid = f'{api_name}::{used.pop()}'
        if tid in tools:
            data.append({'query': inst.get('input', ''), 'tool_id': tid, 'api': api_name})
        else:
            skipped += 1

print(f'Total tools indexed: {len(tools)}')
print(f'Usable single-tool queries: {len(data)} (Skipped: {skipped})')
print(f'Unique APIs represented: {len({x["api"] for x in data})}')


## 3. Nearest-Neighbor Distractor Mining
Rather than using synthetic or artificial distractors, we extract hard semantic distractors directly from the existing tool catalogue using embedding similarity. For each tool in a given split, we identify the top-$k$ most similar tools (excluding itself) using cosine similarity under normalized embeddings.


In [ ]:
def mine_distractors(pool_ids, tools_dict, encoder, k=30, batch_size=128):
    """Mines top-k nearest distractor tools in vector space."""
    emb = encoder.encode([tools_dict[tid] for tid in pool_ids], convert_to_tensor=True,
                         normalize_embeddings=True, batch_size=batch_size)
    sim = emb @ emb.T
    sim.fill_diagonal_(-1.0) # exclude self
    top_indices = sim.topk(min(k, len(pool_ids)-1), dim=1).indices.cpu().numpy()
    return {pool_ids[i]: [pool_ids[j] for j in top_indices[i]] for i in range(len(pool_ids))}


## 4. Contrastive Fine-Tuning with InfoNCE Loss
We train the `all-MiniLM-L6-v2` encoder using `MultipleNegativesRankingLoss` (InfoNCE). For each training query, we sample 3 hard negative distractors from its top 10 nearest neighbours:

$$\mathcal{L} = -\log \frac{\exp(s(q, t^+)/\theta)}{\exp(s(q, t^+)/\theta) + \sum_{t^-} \exp(s(q, t^-)/\theta)}$$


In [ ]:
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
from sentence_transformers.training_args import BatchSamplers

def build_contrastive_dataset(train_queries, tools_dict, train_neighbors, num_hard_negatives=3, seed=42):
    rnd = random.Random(seed)
    rows = {'anchor': [], 'positive': [], 'neg1': [], 'neg2': [], 'neg3': []}
    for item in train_queries:
        tid = item['tool_id']
        cands = train_neighbors.get(tid, [])[:10]
        if not cands: continue
        negs = rnd.sample(cands, min(num_hard_negatives, len(cands)))
        while len(negs) < num_hard_negatives: negs.append(cands[0])
        rows['anchor'].append(item['query'])
        rows['positive'].append(tools_dict[tid])
        for j, ng in enumerate(negs, 1): rows[f'neg{j}'].append(tools_dict[ng])
    return Dataset.from_dict(rows)


## 5. Experience Memory & Episode Execution Loop
We define the retrieve–select–execute episode logic:
- Up to $R=3$ attempts per query.
- Hard exclusion of previously attempted tools within the current query episode.
- Configurations:
  - `none`: Stateless retry-only baseline.
  - `E1`: Flat penalty ($\lambda=0.3$).
  - `E12`: Similarity-weighted penalty ($\lambda=0.3, \tau=0.6$).
  - `E123`: Similarity penalty + hard filtering ($s'(q, t) = -999.0$).


In [ ]:
def attempt_query_episode(Q, T, idx_map, cands, qi, mem, lam, tau, mode, R=3):
    sc = Q[qi] @ T[[idx_map[c] for c in cands]].T
    tried, picks = set(), []
    for a in range(1, R + 1):
        s2 = sc.clone()
        for j, c in enumerate(cands):
            if j in tried:
                s2[j] = -999.0
                continue
            if mode != 'none' and c in mem:
                sims = torch.stack(mem[c]) @ Q[qi]
                if mode == 'E1':
                    s2[j] -= lam
                else: # E12 & E123
                    p = sims.max().item()
                    if p >= tau: s2[j] -= lam * p
                    if mode == 'E123' and (sims >= 0.85).any(): s2[j] = -999.0
        chosen = int(s2.argmax().item())
        picks.append(chosen)
        if chosen == 0:
            return a, picks
        tried.add(chosen)
    return None, picks

def run_two_phase_split(queries, pool_ids, neighbors, Q, T, k=30, lam=0.3, tau=0.6, mode='E12', R=3):
    idx_map = {t: i for i, t in enumerate(pool_ids)}
    by_tool = {}
    for i, x in enumerate(queries):
        by_tool.setdefault(x['tool_id'], []).append(i)
    multi = {t: v for t, v in by_tool.items() if len(v) >= 2}

    # Phase 1: Warm-up query per tool, build memory M
    mem, failed = {}, {}
    for t, indices in multi.items():
        cands = [t] + neighbors[t][:k]
        a, picks = attempt_query_episode(Q, T, idx_map, cands, indices[0], {}, 0, 0, 'none', R)
        failed[t] = (picks[0] != 0)
        for j in picks:
            if j != 0:
                mem.setdefault(cands[j], []).append(Q[indices[0]])

    # Phase 2: Test queries with memory locked
    rows = []
    for t, indices in multi.items():
        cands = [t] + neighbors[t][:k]
        for qi in indices[1:]:
            a, picks = attempt_query_episode(Q, T, idx_map, cands, qi, mem if mode != 'none' else {},
                                            lam, tau, mode, R)
            first_ok = (picks[0] == 0)
            succ_r = (a is not None)
            att = a if a is not None else R
            rows.append((t, qi, bool(failed[t]), first_ok, succ_r, att))
    return rows


## 6. Statistical Hypothesis Testing
- **Paired Exact McNemar Test**: Evaluates whether the number of queries helped by memory differs significantly from the queries regressed.
- **Tool-Cluster Bootstrapped 95% Confidence Intervals (10,000 resamples)**: Resamples at the ground-truth tool level to account for correlation between queries sharing the same tool.


In [ ]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

def cluster_boot(keys, diff, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    _, inv = np.unique(keys, return_inverse=True)
    sums = np.bincount(inv, weights=diff)
    cnt = np.bincount(inv)
    G = len(sums)
    out = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, G, G)
        out[b] = sums[pick].sum() / cnt[pick].sum()
    return np.percentile(out, [2.5, 97.5])

def run_paired_stats(rows_a, rows_b, label='Top-1', mask=None):
    a = np.array([r[3] for r in rows_a], float)
    b = np.array([r[3] for r in rows_b], float)
    keys = np.array([r[0] for r in rows_a])
    if mask is not None:
        a, b, keys = a[mask], b[mask], keys[mask]

    helped = int(((a == 0) & (b == 1)).sum())
    hurt   = int(((a == 1) & (b == 0)).sum())
    both   = int(((a == 1) & (b == 1)).sum())
    neither= int(((a == 0) & (b == 0)).sum())

    p_val = mcnemar([[both, hurt], [helped, neither]], exact=True).pvalue
    lo, hi = cluster_boot(keys, b - a)
    print(f"{label:<25} n={len(a):4d} | Base={a.mean():.3f} -> Mem={b.mean():.3f} (diff {b.mean()-a.mean():+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}], p={p_val:.4f}, helped={helped}, hurt={hurt})")


## 7. Results Visualization (Figure 2 Reproduction)
Below we visualize the experimental results from the paper (Figure 2):
1. **Familiar-error Top-1** across +10, +20, and +30 distractors.
2. **Overall Top-1** across distractors.
3. **Memory Pollution Vulnerability** under 0%–60% noise.


In [ ]:
import os
from IPython.display import Image, display

fig_path = '../figures/results.png' if os.path.exists('../figures/results.png') else 'figures/results.png'
if os.path.exists(fig_path):
    print('Displaying published experimental results (Figure 2):')
    display(Image(fig_path))
else:
    print('Figure 2 is located at figures/results.png')


## 8. Summary of Findings & Citation

| Hypothesis | Description | Result |
| :--- | :--- | :--- |
| **H1** | Distractor APIs degrade baseline retrieval | **Supported** (pooled Top-1 drops from 0.674 to 0.662) |
| **H2** | Experience memory improves overall performance | **Partly Supported** (+2.8 pts overall Top-1 gain; offset by 2.4-2.8 pt regression on non-familiar queries) |
| **H3** | Memory curbs repeated errors on familiar tools | **Solidly Supported** for Cosine Selector (+12.6 pts, 95% CI [+7.8, +18.0]); **Not supported** for LLM Selector |
| **Pollution** | Susceptibility to non-semantic failures (timeouts/policy) | **Confirmed** (drops below no-memory baseline at 20% corruption) |

### BibTeX Citation
```bibtex
@article{farouqe2026tool,
  title={Tool Retrieval under Distractor APIs with Memory-Based Recovery},
  author={Hemal Farouqe, Sheikh Mainul Islam, Tanjida Sultana},
  journal={Academic Research},
  year={2026}
}
```
